# Credit risk scoring: SQL feature engineering, LightGBM, SHAP and a business decision rule

**Goal:** predict the probability that a loan applicant will default (PD) and turn it into an **accept / reject decision**
that a bank could actually use.

What this notebook covers, end to end:

1. **Feature engineering in SQL with DuckDB**: 7 history tables (58 million rows) aggregated into 51 client-level features,
   queried directly from the CSV files.
2. **LightGBM** with hyperparameters tuned beforehand with Optuna (30 trials, cross-validation).
3. **Evaluation the way banks do it**: AUC, Gini, KS, and a **calibration** check (are the predicted PDs right?).
4. **Explainability** with SHAP: global effects and the reasons behind an individual decision.
5. **From PD to a decision**: a profit-based acceptance threshold, a credit score in points, and risk grades A to E.
6. A **submission** file for the leaderboard.

The full project (tests, WoE/IV scorecard, FastAPI service, Streamlit demo, Docker, CI) is on GitHub:
**[github.com/MKkevin2712/credit-risk-scoring](https://github.com/MKkevin2712/credit-risk-scoring)**

## 1. Setup

In [ ]:
import glob
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

try:
    import duckdb
except ImportError:  # DuckDB is not always preinstalled
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

SEED = 42
pd.set_option("display.max_columns", 40)

# Locate the competition files (works on Kaggle and locally)
candidates = glob.glob("/kaggle/input/**/application_train.csv", recursive=True) + glob.glob("data/raw/application_train.csv") + glob.glob("../data/raw/application_train.csv")
DATA = os.path.dirname(candidates[0])
print("Data folder:", DATA)

DuckDB queries the CSV files **directly**, without loading them into pandas first. Each table becomes a SQL view.
Train and test applications are stacked into a single `applications` view (`TARGET` is empty for the test set),
so the exact same SQL builds the features for both.

In [ ]:
con = duckdb.connect()
tables = ["application_train", "application_test", "bureau", "bureau_balance", "previous_application",
          "installments_payments", "credit_card_balance", "POS_CASH_balance"]
for t in tables:
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM read_csv_auto('{DATA}/{t}.csv')")
con.execute("CREATE VIEW applications AS SELECT * FROM application_train UNION ALL BY NAME SELECT * FROM application_test")

pd.DataFrame(
    [(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]) for t in tables], columns=["table", "rows"]
).style.format({"rows": "{:,}"})

## 2. A few facts that drive the modelling choices

In [ ]:
con.execute('''
    SELECT
        ROUND(AVG(TARGET) * 100, 2)                                   AS default_rate_pct,
        ROUND(AVG((DAYS_EMPLOYED = 365243)::INT) * 100, 1)            AS days_employed_365243_pct,
        ROUND(AVG(TARGET) FILTER (WHERE DAYS_EMPLOYED = 365243) * 100, 2)  AS default_rate_if_365243_pct,
        ROUND(AVG(TARGET) FILTER (WHERE DAYS_EMPLOYED <> 365243) * 100, 2) AS default_rate_otherwise_pct
    FROM application_train
''').df()

- Only **~8 % of loans default**: a model predicting "repaid" for everyone would be 92 % accurate and useless.
  We therefore use **AUC, Gini (= 2 × AUC − 1) and KS**, the standard credit scoring metrics.
- `DAYS_EMPLOYED = 365243` (1,000 years!) is a **code**, not a duration: it flags 18 % of applicants, almost all **pensioners**,
  who default less. We replace it with a missing value and keep a 0/1 flag.
- `CODE_GENDER` is a sensitive attribute: it is **excluded** from the model.

## 3. Feature engineering in SQL

Each history table has **several rows per client** (one per past loan, per month, per instalment…).
Each query summarises them into **one row per client** with `GROUP BY`; everything is then joined to the application.

Two details that matter:
- **Partial payments**: an instalment can be paid in several transfers, so payments are first grouped **per instalment**
  before computing lateness; otherwise you count fake late payments.
- **Zero vs unknown**: a client with no credit card has **0 cards** (a count), but an **unknown** card utilisation (not 0 %).

In [ ]:
QUERIES = {
    "application": "-- Application: cleaning and business ratios (one row per client).\n-- Fixes found during exploratory analysis.\nSELECT\n    * EXCLUDE (CODE_GENDER, DAYS_EMPLOYED),   -- CODE_GENDER: sensitive attribute, excluded\n\n    -- 365243 = code for 'no declared employment' (mostly pensioners), not a duration\n    CASE WHEN DAYS_EMPLOYED = 365243 THEN NULL ELSE DAYS_EMPLOYED END AS DAYS_EMPLOYED,\n    (DAYS_EMPLOYED = 365243)::INTEGER                          AS DAYS_EMPLOYED_ANOM,\n\n    -DAYS_BIRTH / 365.25                                       AS AGE_YEARS,\n    AMT_CREDIT  / NULLIF(AMT_INCOME_TOTAL, 0)                  AS CREDIT_INCOME_RATIO,\n    AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL, 0)                  AS ANNUITY_INCOME_RATIO,   -- debt-to-income\n    AMT_ANNUITY / NULLIF(AMT_CREDIT, 0)                        AS CREDIT_TERM,            -- inverse of loan duration\n    AMT_CREDIT  / NULLIF(AMT_GOODS_PRICE, 0)                   AS CREDIT_GOODS_RATIO,     -- financed share\n    AMT_INCOME_TOTAL / NULLIF(CNT_FAM_MEMBERS, 0)              AS INCOME_PER_PERSON,\n    -- share of life in current job; the raw column still contains 365243\n    NULLIF(DAYS_EMPLOYED, 365243) / NULLIF(DAYS_BIRTH, 0)     AS EMPLOYED_AGE_RATIO\nFROM applications",
    "bureau": "-- Client's loans with other lenders (credit bureau).\n-- bureau_balance.STATUS: '0' = up to date, '1'..'5' = increasing delinquency (5 = 120+ days or written off),\n-- 'C' = closed, 'X' = unknown.\nWITH balance_by_credit AS (\n    SELECT\n        SK_ID_BUREAU,\n        COUNT(*) FILTER (WHERE STATUS IN ('0', '1', '2', '3', '4', '5'))     AS n_known_months,\n        COUNT(*) FILTER (WHERE STATUS IN ('1', '2', '3', '4', '5'))          AS n_late_months,\n        MAX(CASE WHEN STATUS IN ('1', '2', '3', '4', '5') THEN STATUS::INTEGER ELSE 0 END) AS worst_status\n    FROM bureau_balance\n    GROUP BY SK_ID_BUREAU\n)\nSELECT\n    b.SK_ID_CURR,\n    COUNT(*)                                                       AS BUREAU_N_CREDITS,\n    COUNT(*) FILTER (WHERE b.CREDIT_ACTIVE = 'Active')             AS BUREAU_N_ACTIVE,\n    AVG((b.CREDIT_ACTIVE = 'Active')::INTEGER)                     AS BUREAU_ACTIVE_RATIO,\n    COUNT(*) FILTER (WHERE b.CREDIT_ACTIVE = 'Bad debt')           AS BUREAU_N_BAD_DEBT,\n    AVG((b.CREDIT_TYPE = 'Credit card')::INTEGER)                  AS BUREAU_CREDIT_CARD_RATIO,\n    SUM(b.AMT_CREDIT_SUM)                                          AS BUREAU_TOTAL_CREDIT,\n    SUM(b.AMT_CREDIT_SUM_DEBT)                                     AS BUREAU_TOTAL_DEBT,\n    SUM(b.AMT_CREDIT_SUM_DEBT) / NULLIF(SUM(b.AMT_CREDIT_SUM), 0)  AS BUREAU_DEBT_CREDIT_RATIO,\n    SUM(b.AMT_CREDIT_SUM_OVERDUE)                                  AS BUREAU_TOTAL_OVERDUE,\n    MAX(b.CREDIT_DAY_OVERDUE)                                      AS BUREAU_MAX_DAYS_OVERDUE,\n    MAX(b.AMT_CREDIT_MAX_OVERDUE)                                  AS BUREAU_MAX_AMT_OVERDUE,\n    SUM(b.CNT_CREDIT_PROLONG)                                      AS BUREAU_N_PROLONGATIONS,\n    MIN(b.DAYS_CREDIT)                                             AS BUREAU_OLDEST_CREDIT_DAYS,\n    MAX(b.DAYS_CREDIT)                                             AS BUREAU_LAST_CREDIT_DAYS,\n    -- monthly repayment history\n    SUM(bb.n_late_months) / NULLIF(SUM(bb.n_known_months), 0)      AS BUREAU_LATE_MONTHS_RATIO,\n    MAX(bb.worst_status)                                           AS BUREAU_WORST_STATUS\nFROM bureau AS b\nLEFT JOIN balance_by_credit AS bb USING (SK_ID_BUREAU)\nGROUP BY b.SK_ID_CURR",
    "previous_application": "-- Client's previous applications at Home Credit.\nSELECT\n    SK_ID_CURR,\n    COUNT(*)                                                       AS PREV_N_APPLICATIONS,\n    AVG((NAME_CONTRACT_STATUS = 'Approved')::INTEGER)              AS PREV_APPROVED_RATIO,\n    AVG((NAME_CONTRACT_STATUS = 'Refused')::INTEGER)               AS PREV_REFUSED_RATIO,\n    AVG((NAME_CONTRACT_STATUS = 'Canceled')::INTEGER)              AS PREV_CANCELED_RATIO,\n    -- granted / requested amount: < 1 when the lender granted less than asked\n    AVG(AMT_CREDIT / NULLIF(AMT_APPLICATION, 0))                   AS PREV_CREDIT_APPLICATION_RATIO,\n    AVG(AMT_CREDIT)                                                AS PREV_MEAN_CREDIT,\n    AVG(CNT_PAYMENT)                                               AS PREV_MEAN_N_PAYMENTS,\n    MAX(DAYS_DECISION)                                             AS PREV_LAST_DECISION_DAYS,\n    MAX(DAYS_DECISION) FILTER (WHERE NAME_CONTRACT_STATUS = 'Refused') AS PREV_LAST_REFUSAL_DAYS\nFROM previous_application\nGROUP BY SK_ID_CURR",
    "installments": "-- Repayments of previous Home Credit loans.\n-- An instalment can be paid in several transfers: group payments per instalment first.\nWITH by_instalment AS (\n    SELECT\n        SK_ID_CURR,\n        SK_ID_PREV,\n        NUM_INSTALMENT_NUMBER,\n        NUM_INSTALMENT_VERSION,\n        MAX(DAYS_INSTALMENT)                       AS due_day,\n        MAX(DAYS_ENTRY_PAYMENT)                    AS last_payment_day,\n        MAX(AMT_INSTALMENT)                        AS amount_due,\n        SUM(AMT_PAYMENT)                           AS amount_paid\n    FROM installments_payments\n    GROUP BY ALL\n),\nenriched AS (\n    SELECT\n        *,\n        GREATEST(last_payment_day - due_day, 0)    AS days_late,\n        -- round to the cent: floating-point noise is not an unpaid amount\n        GREATEST(ROUND(amount_due - amount_paid, 2), 0) AS amount_unpaid\n    FROM by_instalment\n)\nSELECT\n    SK_ID_CURR,\n    COUNT(*)                                       AS INST_N_INSTALMENTS,\n    AVG((days_late > 0)::INTEGER)                  AS INST_LATE_RATIO,\n    AVG((days_late > 30)::INTEGER)                 AS INST_LATE_30_RATIO,\n    AVG(days_late)                                 AS INST_MEAN_DAYS_LATE,\n    MAX(days_late)                                 AS INST_MAX_DAYS_LATE,\n    AVG((amount_unpaid > 0)::INTEGER)              AS INST_UNDERPAID_RATIO,\n    SUM(amount_unpaid)                             AS INST_TOTAL_UNPAID,\n    -- recent behaviour: instalments of the last year\n    AVG((days_late > 0)::INTEGER) FILTER (WHERE due_day >= -365) AS INST_LATE_RATIO_LAST_YEAR\nFROM enriched\nGROUP BY SK_ID_CURR",
    "credit_card": "-- Monthly credit card balances at Home Credit. SK_DPD = days past due.\nSELECT\n    SK_ID_CURR,\n    COUNT(DISTINCT SK_ID_PREV)                                     AS CC_N_CARDS,\n    AVG(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0))          AS CC_MEAN_UTILIZATION,   -- credit limit utilisation\n    MAX(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0))          AS CC_MAX_UTILIZATION,\n    AVG(AMT_DRAWINGS_ATM_CURRENT)                                  AS CC_MEAN_ATM_DRAWINGS,  -- cash withdrawals\n    AVG((SK_DPD > 0)::INTEGER)                                     AS CC_LATE_MONTHS_RATIO,\n    MAX(SK_DPD)                                                    AS CC_MAX_DPD\nFROM credit_card_balance\nGROUP BY SK_ID_CURR",
    "pos_cash": "-- Monthly balances of consumer (POS) and cash loans at Home Credit.\nWITH by_loan AS (\n    -- one row per loan; keep the share left to pay at its latest statement\n    SELECT\n        SK_ID_CURR,\n        SK_ID_PREV,\n        ARG_MAX(CNT_INSTALMENT_FUTURE / NULLIF(CNT_INSTALMENT, 0), MONTHS_BALANCE) AS remaining_ratio,\n        AVG((SK_DPD > 0)::INTEGER)                                                 AS late_months_ratio,\n        COUNT(*)                                                                   AS n_months,\n        MAX(SK_DPD)                                                                AS max_dpd\n    FROM POS_CASH_balance\n    GROUP BY SK_ID_CURR, SK_ID_PREV\n)\nSELECT\n    SK_ID_CURR,\n    COUNT(*)                                                           AS POS_N_LOANS,\n    SUM(late_months_ratio * n_months) / SUM(n_months)                  AS POS_LATE_MONTHS_RATIO,\n    MAX(max_dpd)                                                       AS POS_MAX_DPD,\n    AVG(remaining_ratio)                                               AS POS_REMAINING_RATIO\nFROM by_loan\nGROUP BY SK_ID_CURR"
}

print(QUERIES["installments"])

In [ ]:
for name, query in QUERIES.items():
    con.execute(f"CREATE OR REPLACE TEMP VIEW f_{name} AS {query}")

history = [n for n in QUERIES if n != "application"]
count_cols = [
    c for n in history for (c, *_) in con.execute(f"DESCRIBE f_{n}").fetchall()
    if c.startswith(("BUREAU_N_", "PREV_N_", "INST_N_", "CC_N_", "POS_N_"))
]
joins = "\n".join(f"LEFT JOIN f_{n} USING (SK_ID_CURR)" for n in history)
replace = ", ".join(f"COALESCE({c}, 0) AS {c}" for c in count_cols)   # no history = 0 loans, not unknown

df = con.execute(f"SELECT * REPLACE ({replace}) FROM f_application {joins}").df()
print(f"{df.shape[0]:,} applications x {df.shape[1]} columns")

In [ ]:
# Text columns -> pandas categories (LightGBM handles them natively)
ID, TARGET = "SK_ID_CURR", "TARGET"
features = [c for c in df.columns if c not in (ID, TARGET)]
for c in features:
    if df[c].dtype == object or str(df[c].dtype).startswith("str"):
        df[c] = df[c].astype("category")
    elif str(df[c].dtype) in ("Int64", "Int32", "boolean"):
        df[c] = df[c].astype("float64")

train = df[df[TARGET].notna()].reset_index(drop=True)
test = df[df[TARGET].isna()].reset_index(drop=True)
y = train[TARGET].astype(int)
print(f"Train: {len(train):,} · Test (Kaggle): {len(test):,} · Features: {len(features)}")

In [ ]:
# Which engineered features carry signal? (correlation with default, train only)
new = [c for c in features if c.split("_")[0] in ("BUREAU", "PREV", "INST", "CC", "POS")]
corr = train[new].corrwith(y).dropna().sort_values()
ax = pd.concat([corr.head(6), corr.tail(8)]).plot.barh(figsize=(7, 5), color="#4C72B0")
ax.set_title("Engineered history features most correlated with default"); ax.axvline(0, color="k", lw=0.8)
plt.tight_layout(); plt.show()

## 4. LightGBM

The hyperparameters below were found with **Optuna** (30 trials, 3-fold cross-validation on the training split only).
The best configuration uses **very small trees** (9 leaves, at least 621 applicants per leaf) with strong regularisation:
it generalises better than the default settings.

To evaluate honestly, 20 % of the labelled data is held out (stratified) and never used for training or tuning.
The number of trees is chosen by early stopping on a separate validation split.

In [ ]:
PARAMS = {
    "objective": "binary", "metric": "auc", "learning_rate": 0.03, "bagging_freq": 1,
    "seed": SEED, "verbose": -1,
    "num_leaves": 9,
    "min_child_samples": 621,
    "feature_fraction": 0.36109,
    "bagging_fraction": 0.955736,
    "lambda_l1": 4.634756,
    "lambda_l2": 0.171192,
    "min_split_gain": 0.850729,
}

def metrics(y_true, score):
    auc = roc_auc_score(y_true, score)
    fpr, tpr, _ = roc_curve(y_true, score)
    return {"AUC": auc, "Gini": 2 * auc - 1, "KS": float(np.max(tpr - fpr))}

X_dev, X_hold, y_dev, y_hold = train_test_split(train[features], y, test_size=0.2, stratify=y, random_state=SEED)
X_fit, X_val, y_fit, y_val = train_test_split(X_dev, y_dev, test_size=0.15, stratify=y_dev, random_state=SEED)

model = lgb.train(
    PARAMS, lgb.Dataset(X_fit, y_fit), num_boost_round=6000,
    valid_sets=[lgb.Dataset(X_val, y_val)], callbacks=[lgb.early_stopping(150, verbose=False)],
)
best_rounds = model.best_iteration
pd_hold = model.predict(X_hold, num_iteration=best_rounds)
print(f"Trees: {best_rounds}")
pd.DataFrame({"Hold-out (never seen)": metrics(y_hold, pd_hold)}).T.round(3)

A Gini of about **0.57** on unseen data. For reference, the WoE/IV **scorecard** built in the GitHub project
(19 variables, logistic regression: the traditional, fully auditable bank model) reaches a Gini of about 0.52.

## 5. Are the probabilities right? (calibration)

AUC tells whether the model **ranks** clients well; calibration tells whether its **probabilities** are right.
Banks need both: the PD feeds expected-loss and capital calculations (Basel III).

In [ ]:
deciles = pd.qcut(pd_hold, 10, labels=[f"D{i}" for i in range(1, 11)])
calib = pd.DataFrame({"predicted PD": pd_hold, "observed default rate": y_hold.values}).groupby(deciles, observed=True).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
calib.plot.bar(ax=axes[0], color=["#9DB4D4", "#C44E52"], rot=0)
axes[0].set_title("Predicted vs observed default rate by risk decile"); axes[0].yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes[1].plot(calib["predicted PD"], calib["observed default rate"], "o-", color="#55A868")
axes[1].plot([0, 0.35], [0, 0.35], "k--", lw=0.8); axes[1].set_xlabel("Predicted PD"); axes[1].set_ylabel("Observed default rate")
axes[1].set_title("Calibration curve")
plt.tight_layout(); plt.show()
print(f"Mean predicted PD: {pd_hold.mean():.2%} · observed default rate: {y_hold.mean():.2%}")

The points sit on the diagonal: **no recalibration needed**. This comes from training with the log-loss objective
and **without** rebalancing the classes (oversampling or `scale_pos_weight` would distort the PDs).

## 6. Explainability with SHAP

For each applicant, SHAP splits the predicted risk into one contribution per feature:
risk = average risk + Σ contributions. LightGBM computes exact SHAP values natively (`pred_contrib=True`).

In [ ]:
sample = X_hold.sample(2000, random_state=0)
contrib = model.predict(sample, num_iteration=best_rounds, pred_contrib=True)
data = sample.apply(lambda s: s.cat.codes.where(s.notna()) if str(s.dtype) == "category" else s).astype(float)
explanation = shap.Explanation(values=contrib[:, :-1], base_values=contrib[:, -1], data=data.values,
                               display_data=sample.values, feature_names=features)
shap.plots.beeswarm(explanation, max_display=15, show=False)
plt.title("Feature effects on the risk of default (SHAP)"); plt.tight_layout(); plt.show()

Each dot is an applicant; colour is the feature value (red = high). High external scores (`EXT_SOURCE_*`) push risk down,
recent late payments (`INST_LATE_RATIO_LAST_YEAR`, built in SQL) push it up.
SQL-engineered features such as `CREDIT_TERM` (annuity / credit) and `CREDIT_GOODS_RATIO` rank among the top drivers.

In [ ]:
# Explaining one individual decision: the riskiest applicant of the sample
i = int(np.argmax(contrib.sum(axis=1)))
shap.plots.waterfall(explanation[i], max_display=10, show=False)
plt.title(f"Riskiest applicant of the sample · PD = {1 / (1 + np.exp(-contrib[i].sum())):.1%}"); plt.tight_layout(); plt.show()

## 7. From a probability to a decision

A model only creates value once it drives a decision. With two (illustrative) assumptions:
- a repaid loan earns a **margin of 10 %** of the amount,
- a defaulted loan loses **60 %** of the amount (LGD, *loss given default*),

the expected profit of a loan is $(1 - PD) \times 10\% - PD \times 60\%$, which is positive as long as
$$PD < \frac{10\%}{10\% + 60\%} \approx 14.3\%$$

In [ ]:
MARGIN, LGD = 0.10, 0.60
threshold = MARGIN / (MARGIN + LGD)
amount = X_hold["AMT_CREDIT"].to_numpy()
gain_per_loan = np.where(y_hold.values == 1, -LGD * amount, MARGIN * amount)

def policy(t):
    accepted = pd_hold < t
    return {
        "PD threshold": t,
        "acceptance rate": accepted.mean(),
        "default rate of accepted": y_hold.values[accepted].mean(),
        "bad payers rejected": (~accepted & (y_hold.values == 1)).sum() / (y_hold == 1).sum(),
        "good payers rejected": (~accepted & (y_hold.values == 0)).sum() / (y_hold == 0).sum(),
        "profit": gain_per_loan[accepted].sum(),
    }

table = pd.DataFrame([policy(t) for t in [0.05, 0.08, 0.10, 0.12, threshold, 0.20, 0.30, 1.0 + 1e-9]])
table["profit vs accept all"] = table["profit"] / gain_per_loan.sum() - 1
table.style.format({"PD threshold": "{:.1%}", "acceptance rate": "{:.1%}", "default rate of accepted": "{:.1%}",
                    "bad payers rejected": "{:.1%}", "good payers rejected": "{:.1%}", "profit": "{:,.0f}",
                    "profit vs accept all": "{:+.1%}"})

In [ ]:
grid = np.linspace(0.01, 0.4, 80)
profit = [gain_per_loan[pd_hold < t].sum() for t in grid]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid, np.array(profit) / 1e9, color="#55A868", lw=2, label="Portfolio profit")
ax.axhline(gain_per_loan.sum() / 1e9, color="grey", ls=":", label="Accept everyone")
ax.axvline(threshold, color="k", ls="--", lw=0.8, label=f"Break-even threshold ({threshold:.1%})")
ax.set_xlabel("PD threshold (reject above)"); ax.set_ylabel("Profit (billions)"); ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Hold-out portfolio profit by acceptance threshold"); ax.legend(); plt.tight_layout(); plt.show()

The theoretical break-even threshold lands right on the flat top of the observed profit curve, **because the PDs are
well calibrated**. Compared with accepting everyone, the rule rejects about half of the bad payers for roughly 13 % of
the good ones and raises portfolio profit by about **20 %**.

### Credit score in points and risk grades

The usual scorecard convention: **600 points = odds of 50 good for 1 bad** (PD ≈ 2 %), and **+20 points doubles the odds** (PDO).

In [ ]:
FACTOR = 20 / np.log(2)
OFFSET = 600 - FACTOR * np.log(50)
to_score = lambda p: OFFSET + FACTOR * np.log((1 - p) / p)

bands = [("A", 0.0, 0.02), ("B", 0.02, 0.05), ("C", 0.05, 0.10), ("D", 0.10, threshold), ("E (rejected)", threshold, 1.0)]
grade = pd.cut(pd_hold, [b[1] for b in bands] + [1.0], labels=[b[0] for b in bands], include_lowest=True, right=False)
pd.DataFrame({
    "min score": [to_score(b[2]) if b[2] < 1 else -np.inf for b in bands],
    "share of applicants": pd.Series(grade).value_counts(normalize=True).reindex([b[0] for b in bands]).values,
    "observed default rate": pd.Series(y_hold.values).groupby(np.asarray(grade)).mean().reindex([b[0] for b in bands]).values,
}, index=[b[0] for b in bands]).style.format({"min score": "{:.0f}", "share of applicants": "{:.1%}", "observed default rate": "{:.1%}"})

## 8. Final model and submission

For the leaderboard, the model is retrained on **all** labelled applications, with the number of trees found above
(scaled up slightly since there is more data).

In [ ]:
final = lgb.train(PARAMS, lgb.Dataset(train[features], y), num_boost_round=int(best_rounds * 1.1))
submission = pd.DataFrame({"SK_ID_CURR": test[ID].astype(int), "TARGET": final.predict(test[features])})
submission.to_csv("submission.csv", index=False)
print(submission.shape, f"mean predicted PD: {submission['TARGET'].mean():.2%}")
submission.head()

## Takeaways

- **SQL feature engineering paid off more than hyperparameter tuning**: aggregating the history tables carried the model
  well beyond what the application data alone allows, while Optuna added a few tenths of a point of AUC.
- **Calibration matters as much as AUC** in credit: the PDs are right, so the profit-based threshold works as intended.
- **Explainability is built in**: SHAP gives the reasons behind each decision, which regulators require for a refusal.

The complete project, with a WoE/IV scorecard and its points table, a FastAPI service, a Streamlit demo, Docker and
GitHub Actions CI, is here: **[github.com/MKkevin2712/credit-risk-scoring](https://github.com/MKkevin2712/credit-risk-scoring)**

*Kevin Moukam Nkameni* · if this notebook was useful, an upvote is appreciated!